# 🏦 Bank Personal Loan Granting — Machine Learning Analysis

**Course:** BDA401 Machine Learning Project  
**Dataset:** Bank_Loan_Granting.csv (5,000 bank customers)  
**Goal:** Predict whether a customer will be **granted** a Personal Loan (binary classification: 0 = Not Granted, 1 = Granted)

---

## Background

The dataset contains demographic and banking information for 5,000 bank customers. Roughly **9.6%** of customers were granted a personal loan.  
Our goal is to build a supervised machine learning model that, given an applicant's profile, can accurately predict whether the loan will be **granted** — and to fully explain *why* the model makes each decision.

### Feature Descriptions

| Feature | Description |
|---------|-------------|
| `Age` | Customer age (years) |
| `Experience` | Years of professional experience |
| `Income` | Annual income ($ thousands) |
| `Family` | Family size (1–4) |
| `CCAvg` | Average monthly credit-card spend ($ thousands) |
| `Education` | 1 = Undergrad, 2 = Graduate, 3 = Advanced/Professional |
| `Mortgage` | Value of home mortgage, if any ($ thousands) |
| `Securities Account` | Does the customer hold a securities account? (0/1) |
| `CD Account` | Does the customer hold a CD account? (0/1) |
| `Online` | Does the customer use internet banking? (0/1) |
| `CreditCard` | Does the customer use a bank-issued credit card? (0/1) |
| **`Personal Loan`** | **TARGET — Was the loan granted to the customer? (0 = Not Granted, 1 = Granted)** |

---
## Section 1 — Import Required Libraries

We import all necessary libraries up front:
- **pandas / numpy** — data manipulation  
- **matplotlib / seaborn** — visualisation  
- **scikit-learn** — preprocessing, modelling, and evaluation  
- **ipywidgets** — interactive in-notebook UI  
- **joblib** — saving / loading trained models

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data handling ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})
sns.set_theme(style="whitegrid", palette="muted")

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection     import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing       import StandardScaler, LabelEncoder
from sklearn.ensemble            import RandomForestClassifier
from sklearn.linear_model        import LogisticRegression
from sklearn.tree                import DecisionTreeClassifier, plot_tree
from sklearn.decomposition       import PCA
from sklearn.metrics             import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score, f1_score,
    precision_score, recall_score
)
from sklearn.inspection          import permutation_importance

# ── Model persistence ─────────────────────────────────────────────────────────
import joblib

# ── Interactive widgets ───────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✅  All libraries imported successfully.")

---
## Section 2 — Load and Explore the Dataset

Key things to check on first load:
1. Shape (rows × columns)  
2. Data types — are any columns read incorrectly?  
3. Missing values  
4. Basic statistics (mean, std, quantiles)

In [ ]:
# ── Load raw CSV ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv("Bank_Loan_Granting.csv")
print(f"Shape: {df_raw.shape}")
df_raw.head(10)

In [ ]:
# ── Data types, nulls, stats ──────────────────────────────────────────────────
print("=== Data Types ===")
print(df_raw.dtypes)
print("\n=== Missing Values ===")
print(df_raw.isnull().sum())
print("\n=== Basic Statistics ===")
df_raw.describe()

In [ ]:
# ── Target class distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df_raw["Personal Loan"].value_counts()
axes[0].pie(counts, labels=["Not Granted (0)", "Granted (1)"],
            colors=["#e74c3c", "#27ae60"], autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[0].set_title("Loan Granting Outcome Distribution", fontsize=13, fontweight="bold")

# Count bar
sns.countplot(x="Personal Loan", data=df_raw, palette=["#e74c3c", "#27ae60"], ax=axes[1])
axes[1].set_xticklabels(["Not Granted (0)", "Granted (1)"])
axes[1].set_title("Loan Granting Outcome Counts", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Personal Loan")
axes[1].set_ylabel("Count")
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 30,
                 f"{int(bar.get_height()):,}", ha="center", fontsize=11)

plt.suptitle("⚠️  Class Imbalance: ~9.6% granted  (important for model tuning)",
             fontsize=11, color="#555")
plt.tight_layout()
plt.show()

In [ ]:
# ── Continuous feature distributions by loan granting status ──────────────────
cont_features = ["Age", "Experience", "Income", "CCAvg", "Mortgage"]

# Fix CCAvg first for raw df visualisation
df_vis = df_raw.copy()
df_vis["CCAvg"] = df_vis["CCAvg"].astype(str).str.replace("/", ".", regex=False).astype(float)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(cont_features):
    not_granted = df_vis.loc[df_vis["Personal Loan"] == 0, feat]
    granted     = df_vis.loc[df_vis["Personal Loan"] == 1, feat]
    axes[i].hist(not_granted, bins=30, alpha=0.6, color="#e74c3c", label="Not Granted")
    axes[i].hist(granted,     bins=30, alpha=0.6, color="#27ae60", label="Granted")
    axes[i].set_title(f"Distribution of {feat}", fontweight="bold")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Frequency")
    axes[i].legend()

# Correlation heatmap in last subplot
numeric_cols = ["Age", "Experience", "Income", "Family", "CCAvg",
                "Education", "Mortgage", "Securities Account",
                "CD Account", "Online", "CreditCard", "Personal Loan"]
corr = df_vis[numeric_cols].corr()
sns.heatmap(corr[["Personal Loan"]].sort_values("Personal Loan", ascending=False).drop("Personal Loan"),
            annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            vmin=-1, vmax=1, ax=axes[5], cbar=True)
axes[5].set_title("Correlation with Target (Loan Granted)", fontweight="bold")

plt.suptitle("Feature Distributions — Split by Loan Granting Outcome", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## Section 3 — Data Preprocessing & Feature Engineering

Steps performed:
1. **Fix `CCAvg`** — values stored as `"1/60"` (slash as decimal separator); we replace `"/"` with `"."` and cast to float.
2. **Drop irrelevant columns** — `ID` (identifier, no signal) and `ZIP Code` (high-cardinality postal code with no engineered encoding).
3. **Check for negative `Experience`** — some rows may have negative experience (data entry error); we clip to 0.
4. **Scale features** — `StandardScaler` (mean=0, std=1) for models sensitive to scale (Logistic Regression).  Random Forest is scale-invariant, but we scale anyway for unified pipelines.

In [ ]:
FEATURE_COLS = [
    "Age", "Experience", "Income", "Family", "CCAvg",
    "Education", "Mortgage", "Securities Account", "CD Account", "Online", "CreditCard"
]
TARGET_COL = "Personal Loan"

# ── Preprocessing pipeline ────────────────────────────────────────────────────
df = df_raw.copy()

# Fix CCAvg encoding
df["CCAvg"] = df["CCAvg"].astype(str).str.replace("/", ".", regex=False).astype(float)

# Drop non-predictive columns
df.drop(columns=["ID", "ZIP Code"], inplace=True)

# Clip negative experience values (data artefact)
df["Experience"] = df["Experience"].clip(lower=0)

# Verify no missing values remain
assert df.isnull().sum().sum() == 0, "Unexpected missing values!"

print("✅  Preprocessing complete")
print(f"   Dataset shape : {df.shape}")
print(f"   Feature range :\n{df[FEATURE_COLS].agg(['min','max']).T}")

X = df[FEATURE_COLS]
y = df[TARGET_COL]

---
## Section 4 — Train-Test Split

We use an **80 / 20** split with `stratify=y` to preserve the class ratio in both sets.

$$P(\text{Granted in test set}) \approx P(\text{Granted in train set}) \approx 9.6\%$$

This prevents the model from being evaluated on a test set with very different class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features (needed for LR; RF uses unscaled)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set : {X_train.shape[0]} samples  "
      f"({y_train.mean()*100:.1f}% granted)")
print(f"Test set     : {X_test.shape[0]} samples  "
      f"({y_test.mean()*100:.1f}% granted)")

---
## Section 5 — Model Training

We train and compare **three supervised classifiers**:

### 5a. Logistic Regression
A linear probabilistic model. The decision rule is based on the **logistic (sigmoid) function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = \beta_0 + \beta_1 x_1 + \cdots + \beta_n x_n$$

- If $\sigma(z) \geq 0.5$, predict **Granted**; otherwise **Not Granted**.  
- The boundary is a **hyperplane** in feature space.

### 5b. Decision Tree
Splits data hierarchically by the feature that reduces **Gini impurity** the most:

$$\text{Gini}(t) = 1 - \sum_{k=0}^{1} p_k^2$$

Each internal node tests a single feature; leaf nodes output a class label.

### 5c. Random Forest ✅ (Primary Model)
An **ensemble of decision trees** trained on bootstrap samples. Final prediction is the majority vote across all trees. Key advantages:
- Handles non-linear relationships naturally  
- Robust to outliers and collinear features  
- Provides feature importances via mean decrease in impurity  
- Much lower variance than a single Decision Tree

In [ ]:
# ── Train all three models ────────────────────────────────────────────────────

lr  = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
dt  = DecisionTreeClassifier(max_depth=6, random_state=42)
rf  = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

# Logistic Regression & DT need scaled data; RF does not require scaling
lr.fit(X_train_sc, y_train)
dt.fit(X_train,    y_train)   # DT is also scale-invariant
rf.fit(X_train,    y_train)

print("✅  All models trained.")

# ── 5-fold cross-validation on RF ────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")
print(f"\nRandom Forest — 5-Fold CV ROC-AUC")
print(f"  Mean  : {cv_scores.mean():.4f}")
print(f"  Std   : {cv_scores.std():.4f}")
print(f"  Scores: {np.round(cv_scores, 4)}")

# Save the primary model
joblib.dump(rf,           "rf_model.pkl")
joblib.dump(scaler,       "scaler.pkl")
joblib.dump(FEATURE_COLS, "feature_cols.pkl")
print("\n✅  Model saved to rf_model.pkl")

---
## Section 6 — Model Evaluation

We use the following metrics for binary classification:

| Metric | Formula | What it tells us |
|--------|---------|-----------------|
| **Accuracy** | $\frac{TP+TN}{TP+TN+FP+FN}$ | Overall correct predictions |
| **Precision** | $\frac{TP}{TP+FP}$ | Of predicted grants, how many were actually granted? |
| **Recall** | $\frac{TP}{TP+FN}$ | Of actual loan grants, how many did the model catch? |
| **F1-Score** | $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ | Harmonic mean of precision and recall |
| **ROC-AUC** | Area under the ROC curve | Discrimination ability across all thresholds |

> 📝 Because only ~9.6% of customers were granted the loan, **ROC-AUC and F1 are more informative than raw accuracy**.

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
lr_pred  = lr.predict(X_test_sc);  lr_prob  = lr.predict_proba(X_test_sc)[:, 1]
dt_pred  = dt.predict(X_test);     dt_prob  = dt.predict_proba(X_test)[:, 1]
rf_pred  = rf.predict(X_test);     rf_prob  = rf.predict_proba(X_test)[:, 1]

# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
model_names = ["Logistic Regression", "Decision Tree", "Random Forest"]
preds       = [lr_pred, dt_pred, rf_pred]

for ax, name, pred in zip(axes, model_names, preds):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Granted", "Granted"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nAccuracy = {accuracy_score(y_test, pred):.3f}", fontweight="bold")

plt.suptitle("Confusion Matrices — Test Set (20%)", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

for name, prob in zip(model_names, [lr_prob, dt_prob, rf_prob]):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, lw=2, label=f"{name}  (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random guess")
ax.fill_between(*roc_curve(y_test, rf_prob)[:2], alpha=0.08, color="green")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curves — Model Comparison", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ── Metrics summary table ─────────────────────────────────────────────────────
rows = []
for name, pred, prob in zip(model_names, preds, [lr_prob, dt_prob, rf_prob]):
    rows.append({
        "Model":     name,
        "Accuracy":  f"{accuracy_score(y_test, pred):.4f}",
        "Precision": f"{precision_score(y_test, pred):.4f}",
        "Recall":    f"{recall_score(y_test, pred):.4f}",
        "F1-Score":  f"{f1_score(y_test, pred):.4f}",
        "ROC-AUC":   f"{roc_auc_score(y_test, prob):.4f}",
    })

metrics_df = pd.DataFrame(rows).set_index("Model")
display(metrics_df.style.highlight_max(axis=0, color="#c8f7c5").set_caption("Model Performance Summary"))

---
## Section 7 — Decision Boundary Visualization

A **decision boundary** is the set of points in feature space where the model's predicted probability equals exactly 50% — i.e. the model is equally likely to predict either class.

$$P(\text{Granted} \mid \mathbf{x}) = 0.5$$

Because our model uses **11 features**, the boundary lives in 11-dimensional space and cannot be plotted directly.  We use two strategies:

### Strategy A — PCA 2-D Projection
We project all data onto the first two **Principal Components** (PC1, PC2), which together capture the maximum possible variance.  The decision boundary is then drawn in this 2-D subspace.

### Strategy B — Two Most Important Features
We hold all other features at their median value and draw the boundary over the grid of `Income` × `CCAvg` — the two strongest predictors of loan granting.

In [ ]:
# ── Strategy A  — PCA 2-D Decision Boundary ──────────────────────────────────

X_all_sc = scaler.transform(X)
pca       = PCA(n_components=2, random_state=42)
X_pca     = pca.fit_transform(X_all_sc)

# Meshgrid in PCA space
h = 0.15
x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Map mesh back to original feature space → get RF probability
mesh_pca  = np.c_[xx.ravel(), yy.ravel()]
mesh_orig = scaler.inverse_transform(pca.inverse_transform(mesh_pca))
Z = rf.predict_proba(mesh_orig)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
cf = ax.contourf(xx, yy, Z, levels=50, cmap="RdYlGn", alpha=0.8)
plt.colorbar(cf, ax=ax, label="P(Loan Granted)", pad=0.02)
ax.contour(xx, yy, Z, levels=[0.5], colors="black",
           linewidths=2, linestyles="--")

# Scatter actual data
y_arr = y.values
c_map = {0: "#c0392b", 1: "#1e8449"}
colors = [c_map[yi] for yi in y_arr]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, s=6, alpha=0.35, linewidths=0)

not_granted_p = mpatches.Patch(color="#c0392b", label="Not Granted (actual)")
granted_p     = mpatches.Patch(color="#1e8449", label="Granted (actual)")
bnd_line      = plt.Line2D([0], [0], color="black", lw=2, ls="--", label="Decision boundary (50%)")
ax.legend(handles=[not_granted_p, granted_p, bnd_line], fontsize=10, loc="upper right")

ev = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1  ({ev[0]*100:.1f}% variance explained)", fontsize=12)
ax.set_ylabel(f"PC2  ({ev[1]*100:.1f}% variance explained)", fontsize=12)
ax.set_title(
    f"Random Forest — Decision Boundary (PCA Projection)\n"
    f"PC1+PC2 capture {sum(ev)*100:.1f}% of total variance | Green = Granted, Red = Not Granted",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print(f"\nPC1 explains {ev[0]*100:.1f}% of variance — mainly driven by Income & CCAvg")
print(f"PC2 explains {ev[1]*100:.1f}% of variance — mainly driven by Age & Experience")
print(f"\nTop PCA component loadings (PC1):")
pc1_loadings = pd.Series(pca.components_[0], index=FEATURE_COLS).abs().sort_values(ascending=False)
print(pc1_loadings.to_string())

In [ ]:
# ── Strategy B  — Income vs CCAvg (2 most important features) ─────────────────
# All other features held at their median value

medians = X.median()
income_range = np.linspace(X["Income"].min(), X["Income"].max(), 200)
ccavg_range  = np.linspace(X["CCAvg"].min(),  X["CCAvg"].max(),  200)
ii, cc = np.meshgrid(income_range, ccavg_range)

grid_df = pd.DataFrame(
    np.tile(medians.values, (ii.size, 1)),
    columns=FEATURE_COLS
)
grid_df["Income"] = ii.ravel()
grid_df["CCAvg"]  = cc.ravel()

Z2 = rf.predict_proba(grid_df)[:, 1].reshape(ii.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability heatmap
cf2 = axes[0].contourf(ii, cc, Z2, levels=50, cmap="RdYlGn", alpha=0.9)
plt.colorbar(cf2, ax=axes[0], label="P(Granted)")
axes[0].contour(ii, cc, Z2, levels=[0.5], colors="black", linewidths=2, linestyles="--")
axes[0].scatter(X.loc[y == 0, "Income"], X.loc[y == 0, "CCAvg"],
                c="#c0392b", s=5, alpha=0.3, label="Not Granted")
axes[0].scatter(X.loc[y == 1, "Income"], X.loc[y == 1, "CCAvg"],
                c="#1e8449", s=8, alpha=0.6, label="Granted")
axes[0].set_xlabel("Annual Income ($K)")
axes[0].set_ylabel("Monthly CC Spend ($K)")
axes[0].set_title("Decision Boundary\nIncome vs CCAvg", fontweight="bold")
axes[0].legend(fontsize=9)

# Decision region (hard boundary)
Z2_binary = (Z2 >= 0.5).astype(int)
cmap_binary = ListedColormap(["#f1948a", "#82e0aa"])
axes[1].contourf(ii, cc, Z2_binary, cmap=cmap_binary, alpha=0.6)
axes[1].contour(ii, cc, Z2, levels=[0.5], colors="black", linewidths=2, linestyles="--")
axes[1].scatter(X.loc[y == 0, "Income"], X.loc[y == 0, "CCAvg"],
                c="#c0392b", s=5, alpha=0.3)
axes[1].scatter(X.loc[y == 1, "Income"], X.loc[y == 1, "CCAvg"],
                c="#1e8449", s=8, alpha=0.6)
axes[1].set_xlabel("Annual Income ($K)")
axes[1].set_ylabel("Monthly CC Spend ($K)")
axes[1].set_title("Hard Decision Regions\n(all other features at median)", fontweight="bold")
not_granted_p = mpatches.Patch(color="#f1948a", alpha=0.6, label="Predicted: Not Granted")
granted_p     = mpatches.Patch(color="#82e0aa", alpha=0.6, label="Predicted: Granted")
axes[1].legend(handles=[not_granted_p, granted_p], fontsize=9)

plt.suptitle("Income × CCAvg — Random Forest Decision Boundary (Loan Granting)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Decision Tree – visual explanation of split rules ─────────────────────────
# (limited to depth=4 for readability)
dt_vis = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    dt_vis,
    feature_names=FEATURE_COLS,
    class_names=["Not Granted", "Granted"],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    proportion=False,
)
ax.set_title(
    "Decision Tree — Depth 4 (simplified)\n"
    "Each node shows: split rule | Gini impurity | sample count | class distribution",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("• Orange nodes  → predict Granted")
print("• Blue-ish nodes → predict Not Granted")
print("• Gini = 0.000  → perfectly pure node (all one class)")
print("• Gini = 0.500  → maximally impure node (50/50 split)")

---
## Section 8 — Feature Importance Analysis

Random Forest provides two kinds of importance:

1. **Gini Importance (MDI)** — Mean decrease in Gini impurity across all trees and splits.  Fast but can overestimate the importance of high-cardinality features.
2. **Permutation Importance** — Measure the drop in accuracy when a feature's values are randomly shuffled.  More reliable but slower.

We also show the **Logistic Regression coefficients** (after scaling) to compare the linear model's understanding of each feature.

In [ ]:
# ── Gini (MDI) feature importances ───────────────────────────────────────────
fi_df = pd.DataFrame({
    "Feature":    FEATURE_COLS,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=True)

# ── Permutation importances ───────────────────────────────────────────────────
perm = permutation_importance(rf, X_test, y_test,
                              n_repeats=20, random_state=42, n_jobs=-1)
pi_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Mean":    perm.importances_mean,
    "Std":     perm.importances_std,
}).sort_values("Mean", ascending=True)

# ── LR coefficients ────────────────────────────────────────────────────────────
lr_coef_df = pd.DataFrame({
    "Feature":    FEATURE_COLS,
    "Coefficient": lr.coef_[0]
}).sort_values("Coefficient")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# MDI
bars_mdi = axes[0].barh(fi_df["Feature"], fi_df["Importance"],
                        color=["#27ae60" if v > 0.05 else "#3498db"
                               for v in fi_df["Importance"]])
axes[0].set_xlabel("Mean Decrease in Impurity")
axes[0].set_title("RF — Gini (MDI) Importance", fontweight="bold")
for bar in bars_mdi:
    axes[0].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f"{bar.get_width():.3f}", va="center", fontsize=8)

# Permutation
colors_perm = ["#27ae60" if v > 0 else "#e74c3c" for v in pi_df["Mean"]]
axes[1].barh(pi_df["Feature"], pi_df["Mean"], xerr=pi_df["Std"],
             color=colors_perm, capsize=4)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("Mean Accuracy Drop when Shuffled")
axes[1].set_title("RF — Permutation Importance", fontweight="bold")

# LR coefficients
bar_colors = ["#27ae60" if c > 0 else "#e74c3c" for c in lr_coef_df["Coefficient"]]
axes[2].barh(lr_coef_df["Feature"], lr_coef_df["Coefficient"], color=bar_colors)
axes[2].axvline(0, color="black", lw=0.8)
axes[2].set_xlabel("Coefficient (after standardisation)")
axes[2].set_title("Logistic Regression Coefficients", fontweight="bold")

plt.suptitle("Feature Importance — Three Views", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\n📋  Top 5 features by Gini Importance:")
print(fi_df.sort_values("Importance", ascending=False).head(5).to_string(index=False))

---
## Section 9 — Interactive Prediction UI (ipywidgets)

Use the sliders and dropdowns below to enter a customer's profile.  
Click **Predict** to see the model's loan decision and confidence score.

> 💡 This widget works in **Jupyter Notebook / JupyterLab / VS Code** (requires `ipywidgets` installed).

In [ ]:
# ── Widget definitions ────────────────────────────────────────────────────────
style  = {"description_width": "150px"}
layout = widgets.Layout(width="400px")

w_age        = widgets.IntSlider(min=18,  max=90,  value=35,  step=1,  description="Age",              style=style, layout=layout)
w_exp        = widgets.IntSlider(min=0,   max=43,  value=10,  step=1,  description="Experience (yrs)", style=style, layout=layout)
w_income     = widgets.IntSlider(min=8,   max=224, value=60,  step=1,  description="Income ($K)",      style=style, layout=layout)
w_family     = widgets.Dropdown( options=[1,2,3,4],            value=2,           description="Family Size",       style=style, layout=layout)
w_ccavg      = widgets.FloatSlider(min=0, max=10,  value=1.5, step=0.1,description="CC Spend ($K/mo)", style=style, layout=layout)
w_education  = widgets.Dropdown( options=[(1,"Undergraduate"),(2,"Graduate"),(3,"Advanced")],
                                  value=2, description="Education",         style=style, layout=layout)
w_mortgage   = widgets.IntSlider(min=0,   max=635, value=0,   step=5,  description="Mortgage ($K)",    style=style, layout=layout)
w_securities = widgets.Checkbox( value=False, description="Securities Account")
w_cd         = widgets.Checkbox( value=False, description="CD Account")
w_online     = widgets.Checkbox( value=True,  description="Online Banking")
w_cc         = widgets.Checkbox( value=False, description="Bank Credit Card")

btn_predict = widgets.Button(
    description="🔍  Predict Loan Decision",
    button_style="primary",
    layout=widgets.Layout(width="260px", height="40px")
)
out = widgets.Output()

# ── Callback ──────────────────────────────────────────────────────────────────
def on_predict(b):
    with out:
        clear_output(wait=True)
        row = pd.DataFrame([[
            w_age.value, w_exp.value, w_income.value, w_family.value, w_ccavg.value,
            w_education.value, w_mortgage.value,
            int(w_securities.value), int(w_cd.value),
            int(w_online.value), int(w_cc.value)
        ]], columns=FEATURE_COLS)

        prob = rf.predict_proba(row)[0][1]
        pred = int(prob >= 0.5)
        label = "✅  LOAN GRANTED" if pred == 1 else "❌  LOAN NOT GRANTED"
        color = "#27ae60"          if pred == 1 else "#e74c3c"

        bar_filled = "█" * int(prob * 40)
        bar_empty  = "░" * (40 - int(prob * 40))

        html = f"""
        <div style="background:{color};color:white;padding:14px 20px;border-radius:10px;
                    font-family:Arial;font-size:18px;font-weight:bold;margin-bottom:10px;">
            Personal Loan Decision: {label}
        </div>
        <div style="font-family:Arial;font-size:14px;padding:8px 0;">
            <b>Probability loan is granted:</b>  {prob*100:.1f}%<br>
            <br>
            <span style="font-family:monospace;font-size:13px;color:{color};">{bar_filled}</span>
            <span style="font-family:monospace;font-size:13px;color:#bbb;">{bar_empty}</span>
            &nbsp; <span style="font-size:12px;color:#555;">(decision threshold = 50%)</span>
        </div>
        <div style="font-family:Arial;font-size:12px;color:#555;margin-top:8px;">
            Model: Random Forest (200 trees) | Trained on 4,000 customers
        </div>
        """
        display(HTML(html))

btn_predict.on_click(on_predict)

# ── Layout ────────────────────────────────────────────────────────────────────
col1 = widgets.VBox([w_age, w_exp, w_income, w_family, w_ccavg, w_education, w_mortgage])
col2 = widgets.VBox([
    widgets.HTML("<b>Bank products held:</b>"),
    w_securities, w_cd, w_online, w_cc,
    widgets.HTML("<br>"),
    btn_predict
])

display(widgets.HTML("<h3 style='font-family:Arial'>🏦 Bank Loan Granting Predictor</h3>"))
display(widgets.HBox([col1, widgets.HTML("&nbsp;&nbsp;&nbsp;"), col2]))
display(out)